# Per-wavelength models + unwrap with a wavefront error besides the pistons

The dispersed-lantern route of `workbench.ipynb` (Part B: one model per wavelength of the
wrapped piston, then the multi-wavelength unwrap), with every per-wavelength model **trained on
signals that carry a wavefront error besides the pistons**: by default an SCAO residual of
120 nm rms (`gen_wfe_spectral.py` with `WFE_PSD=ao`, `wavefront.ao_residual_screens`: corrected
modes below the M4 control frequency, fitting error beyond; the same OPD at every wavelength;
global and petal pistons removed). `WFE_PSD=wfe WFE_RMS=30`: the earlier f^-2 screens. 10000 configurations of the
+-8 lambda spectral range, 8000 for training, 2000 for validation; the 21 wavelengths of the unwrap.

Methods, as at one wavelength: linear, dense polynomials of degree 2 and 3, sparse cubic (OMP),
the closed form inverted with the fixed-step method, an MLP (the 4 x 256 nets of cache_big when
present, else 3 x 128); each + the same unwrap.
(Symbolic regression is not run per wavelength: one search per wavelength and class of piston.)

In [ ]:
import os
os.environ["CBEAM_BACKEND"] = "numpy"
import numpy as np
import matplotlib.pyplot as plt
import ml_reconstruct as ml
from ml_reconstruct.notebook import *
from ml_reconstruct import forward as fw, petals
from ml_reconstruct.simulate import bp
from ml_reconstruct.presets import settings

S = settings(); LANTERN, DATA = S["lantern"], S["data_dir"]
KIND = os.environ.get("WFE_PSD", "ao")          # ao: SCAO residual screens; wfe: f^-2
RMS = float(os.environ.get("WFE_RMS", 120 if KIND == "ao" else 30))
M_MODES = 5
ds = ml.LanternDataset.load(f"{DATA}/spectral_{KIND}{RMS:g}_21wl.npz")
lam = np.asarray(ds.wavelengths_nm, float)
SB_tr, CB_tr = ds.train
SB_va, CB_va = ds.val
n_tr, n_wl, n_cores = SB_tr.shape
xs = ml.Standardizer().fit(SB_tr.reshape(n_tr, -1))
X3_tr = xs.transform(SB_tr.reshape(n_tr, -1)).reshape(SB_tr.shape)
X3_va = xs.transform(SB_va.reshape(len(SB_va), -1)).reshape(SB_va.shape)
SPAN = S["spectral_span_nm"]
_p = bp.get_simulation_parameters(lantern=LANTERN)
_f = bp.load_ifunc(_p["ifunc_file"]); bp.center_ifunc_mask(_f)
_g = petals.petal_geometry(np.asarray(_f.mask_inf_func) > 0, np.asarray(_f.influence_function), 5)
TWIN = petals.twin_matrix(_g["opposite"], _g["ref"])
cache = ml.ModelCache(f"{DATA}/cache_{KIND}{RMS:g}_perlambda")
print(f"{LANTERN}: {n_tr} training / {len(SB_va)} validation samples, {n_wl} wavelengths "
      f"({lam[0]:.0f}-{lam[-1]:.0f} nm) x {n_cores} cores, pistons +-{SPAN:.0f} nm, WFE {RMS:g} nm rms")

In [ ]:
METHODS = [("linear", "ridge", 1), ("dense poly, degree 2", "ridge", 2), ("dense poly, degree 3", "ridge", 3),
           ("sparse poly (OMP)", "omp", 3), ("MLP", "mlp", None), ("closed form", "closed_form", None)]
K_W, SCREEN_W = S["k_wrap"], 2000
WL_ALL, FIB_ALL = np.arange(n_wl), np.arange(n_cores)
RESULTS, DOF, FITS, PHAT = {}, {}, {}, {}
for name, kind, deg in METHODS:
    tag = f"{kind}{'-d%d' % deg if deg and kind == 'ridge' else ''}{'-K%d' % K_W if kind == 'omp' else ''}"
    FITS[name] = ml.fit_wrapped_models(X3_tr, X3_va, CB_tr, CB_va, lam, WL_ALL, FIB_ALL, degree=deg or 3,
                                       K=K_W, screen=SCREEN_W, cache=cache, tag=tag, model=kind, twin=TWIN)
    P_hat, Q = ml.unwrap_all(FITS[name], SPAN)
    PHAT[name] = (P_hat, Q)
    score(RESULTS, f"{name} + unwrap", P_hat, CB_va)
    cache.verbose = False
    per_wl = [cache(f"{tag}-w{w}", None) for w in WL_ALL]
    DOF[f"{name} + unwrap"] = sum((m.n_constants / M_MODES) if kind == "closed_form" else n_dof(m, 2) for m in per_wl)
    cache.verbose = True
    wrong = 100 * np.mean(np.abs(P_hat - CB_va) > lam.min() / 2)
    print(f"   wrapped RMSE per wavelength, mean {FITS[name]['sig_va'].mean():.1f} nm; wrong fringes {wrong:.2f}%")

# --- the MLP per wavelength: the 4x256 nets of cache_big (scripts/train_mlp_big.py and
# continue_mlp_big.py on sauron), in place of the 3x128 above; the unwrap is weighted by their
# training-row errors, as for the other methods.
BIG_DIR = f"{DATA}/cache_big"
if KIND == "ao" and RMS == 120 and os.path.exists(f"{BIG_DIR}/w0-mlp-256x4.npz"):
    gh, cf = np.zeros((len(CB_va), M_MODES, n_wl)), np.zeros((len(CB_va), M_MODES, n_wl))
    sig_tr, sig_va, n_par = np.zeros((n_wl, M_MODES)), np.zeros((n_wl, M_MODES)), 0
    for w, lw in enumerate(lam):
        xw = ml.Standardizer().fit(SB_tr[:, w, :])
        params, layers, _ = ml.load_mlp(f"{BIG_DIR}/w{w}-mlp-256x4.npz")
        gh[:, :, w], cf[:, :, w] = ml.phasor_decode(np.asarray(ml.mlp_forward(params, xw.transform(SB_va[:, w, :]))), lw)
        g_tr = ml.PhasorCodec(lw).inverse(np.asarray(ml.mlp_forward(params, xw.transform(SB_tr[:, w, :]))))
        sig_tr[w] = np.sqrt(np.mean(ml.wrap(g_tr - ml.wrap(CB_tr, lw), lw) ** 2, axis=0))
        sig_va[w] = np.sqrt(np.mean(ml.wrap(gh[:, :, w] - ml.wrap(CB_va, lw), lw) ** 2, axis=0))
        n_par += n_dof(params, 2)
    FITS["MLP"] = dict(ghat_va=gh, conf_va=cf, sig_tr=sig_tr, sig_va=sig_va, periods=lam, wl_use=WL_ALL)
    P_hat, Q = ml.unwrap_all(FITS["MLP"], SPAN)
    PHAT["MLP"] = (P_hat, Q)
    del RESULTS["MLP + unwrap"], DOF["MLP + unwrap"]
    score(RESULTS, "MLP + unwrap", P_hat, CB_va, note="   (4x256 per wavelength)")
    DOF["MLP + unwrap"] = n_par
    wrong = 100 * np.mean(np.abs(P_hat - CB_va) > lam.min() / 2)
    print(f"   wrapped RMSE per wavelength, mean {sig_va.mean():.1f} nm; wrong fringes {wrong:.2f}%")

In [ ]:
# wrapped error of each method's per-wavelength models (mean over the pistons)
plt.figure(figsize=(9, 4))
for name, _, _ in METHODS:
    plt.semilogy(lam, FITS[name]["sig_va"].mean(axis=1), "o-", ms=3, label=name)
plt.xlabel("wavelength (nm)"); plt.ylabel("wrapped RMSE (nm)")
plt.title(f"per-wavelength models with {RMS:g} nm of wavefront error: wrapped error")
plt.grid(alpha=0.3, which="both"); plt.legend(fontsize=8, loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout(); plt.show()

In [ ]:
# the unwrap of the best method
best = min(RESULTS, key=lambda m: RESULTS[m].mean()).replace(" + unwrap", "")
print("best:", best)
P_hat, Q = PHAT[best]
print_unwrap_table(P_hat, CB_va, FITS[best])
plot_unwrap(P_hat, CB_va, Q, SPAN)

In [ ]:
show_results(RESULTS, f"{LANTERN}: 21 wavelengths, +-{SPAN / 1000:g} um, {RMS:g} nm wavefront error, each method + unwrap",
             dof=DOF, sort=True, logy=True)

In [ ]:
# error distributions: the per-wavelength wrapped estimates and the fused one, one piston
k_pist = 2
_ghat = FITS[best]["ghat_va"]                      # (W, S, M) wrapped estimates
_per = np.asarray(FITS[best]["periods"], float)     # wrap period per wavelength
_g = np.asarray(_ghat)
if _g.shape[0] == len(CB_va):
    _est = _g[:, k_pist, :]                  # (samples, wavelengths)
elif _g.shape[1] == len(CB_va):
    _est = np.transpose(_g[:, :, k_pist])    # (wavelengths, samples) -> same
else:
    _est = _g[:, k_pist, :]
err_w = np.array([ml.wrap(_est[:, w] - ml.wrap(CB_va[:, k_pist], _per[w]), _per[w])
                  for w in range(len(_per))])
err_u = P_hat[:, k_pist] - CB_va[:, k_pist]
fig, ax = plt.subplots(figsize=(9, 3.8))
boxes = ax.boxplot(list(err_w) + [err_u], widths=0.65, showfliers=False, patch_artist=True)
for i_b, patch in enumerate(boxes["boxes"]):
    last = i_b == len(err_w)
    patch.set(facecolor="#ff7f0e" if last else "#bcd2e8", edgecolor="#1f4e79" if last else "#4c72b0",
              linewidth=1.0 if last else 0.6)
for med in boxes["medians"]:
    med.set(color="#333333", linewidth=1.0)
ax.axhline(0, color="k", lw=0.6)
ax.set_xticks(range(1, len(err_w) + 2))
ax.set_xticklabels([f"{p:.0f}" for p in _per] + ["unwrapped"], rotation=90, fontsize=7)
ax.set_xlabel("wavelength (nm), then the fused estimate")
ax.set_ylabel(f"error of piston {k_pist} (nm)")
ax.set_title(f"{best}, {RMS:g} nm SCAO residual: each wavelength alone against the unwrap")
plt.tight_layout(); plt.show()
print(f"piston {k_pist}: per-wavelength rms "
      f"{np.sqrt(np.mean(err_w ** 2, axis=1)).mean():.1f} nm, unwrapped rms "
      f"{np.sqrt(np.mean(err_u ** 2)):.2f} nm")
